In [1]:
import sys, json, os
sys.path.insert(0, "../src")
from pathlib import Path
from dotenv import load_dotenv
load_dotenv("../.env")

cases = json.loads(Path("../data/eval_cases.json").read_text())
results = json.loads(Path("../data/run_results.json").read_text())

case_map = {c["id"]: c for c in cases}
print(f"Loaded {len(cases)} cases, {len(results)} run results")

Loaded 11 cases, 11 run results


In [2]:
import pandas as pd
from arxiv_agent.eval import (
    score_tool_called,
    score_correct_paper,
    score_keyword_coverage,
    score_no_hallucination,
)

rows = []
for run in results:
    case = case_map[run["id"]]
    row = {
        "id": run["id"],
        "category": case["category"],
        "question": case["question"][:60],
        "trace_id": run["trace_id"],
        "tool_called": score_tool_called(run, case["expects_tool_call"]),
        "correct_paper": score_correct_paper(run, case.get("expected_paper_id")),
        "keyword_coverage": score_keyword_coverage(run, case.get("expected_keywords", [])),
        "no_hallucination": score_no_hallucination(run, case["expects_tool_call"]),
    }
    rows.append(row)

df = pd.DataFrame(rows)
print(df[["id", "category", "tool_called", "correct_paper", "keyword_coverage", "no_hallucination"]])

       id        category  tool_called  correct_paper  keyword_coverage  \
0   kp_01     known_paper            1            1.0              0.50   
1   kp_02     known_paper            1            1.0              1.00   
2   kp_03     known_paper            1            1.0              1.00   
3   kp_04     known_paper            1            1.0              1.00   
4   sq_01    search_query            1            NaN              0.75   
5   sq_02    search_query            1            NaN              0.75   
6   sq_03    search_query            1            NaN              0.50   
7   nt_01  no_tool_needed            1            NaN              1.00   
8   nt_02  no_tool_needed            1            NaN              1.00   
9   bi_01       bad_input            1            1.0              1.00   
10  bi_02       bad_input            0            NaN              1.00   

    no_hallucination  
0                  1  
1                  1  
2                  1  
3      

In [4]:
from arxiv_agent.observability import get_client, flush

lf = get_client()
score_cols = ["tool_called", "correct_paper", "keyword_coverage", "no_hallucination"]

for _, row in df.iterrows():
    for score_name in score_cols:
        val = row[score_name]
        if val is None:
            continue
        lf.create_score(
            trace_id=row["trace_id"],
            name=score_name,
            value=float(val),
        )

flush()
print("Scores posted to Langfuse. Open the UI to see them on each trace.")
print(f"UI: {os.environ['LANGFUSE_BASE_URL']}")

Scores posted to Langfuse. Open the UI to see them on each trace.
UI: http://localhost:3000


In [5]:
print("=== Overall Scores ===")
for col in score_cols:
    valid = df[col].dropna()
    print(f"  {col}: {valid.mean():.2f} ({valid.sum():.0f}/{len(valid)} pass)")

print("\n=== By Category ===")
for cat in df["category"].unique():
    sub = df[df["category"] == cat]
    print(f"\n  {cat} ({len(sub)} cases):")
    for col in score_cols:
        valid = sub[col].dropna()
        if len(valid) > 0:
            print(f"    {col}: {valid.mean():.2f}")

=== Overall Scores ===
  tool_called: 0.91 (10/11 pass)
  correct_paper: 1.00 (5/5 pass)
  keyword_coverage: 0.86 (10/11 pass)
  no_hallucination: 0.91 (10/11 pass)

=== By Category ===

  known_paper (4 cases):
    tool_called: 1.00
    correct_paper: 1.00
    keyword_coverage: 0.88
    no_hallucination: 1.00

  search_query (3 cases):
    tool_called: 1.00
    keyword_coverage: 0.67
    no_hallucination: 1.00

  no_tool_needed (2 cases):
    tool_called: 1.00
    keyword_coverage: 1.00
    no_hallucination: 1.00

  bad_input (2 cases):
    tool_called: 0.50
    correct_paper: 1.00
    keyword_coverage: 1.00
    no_hallucination: 0.50


In [6]:
base_url = os.environ["LANGFUSE_BASE_URL"]

print("=== Failing Cases ===\n")
for _, row in df.iterrows():
    failures = []
    for col in score_cols:
        if row[col] is not None and row[col] < 1.0:
            failures.append(f"{col}={row[col]:.2f}")
    if failures:
        print(f"[{row['id']}] {row['question']}")
        print(f"  Failed: {', '.join(failures)}")
        print(f"  Trace: {base_url}/trace/{row['trace_id']}")
        print()

=== Failing Cases ===

[kp_01] What is the main contribution of the Attention Is All You Ne
  Failed: keyword_coverage=0.50
  Trace: http://localhost:3000/trace/0dec91584342b01f947c791badf2a8a9

[sq_01] Find a paper on retrieval augmented generation and explain h
  Failed: keyword_coverage=0.75
  Trace: http://localhost:3000/trace/f39a0f96299119da5cb992b2d2ad2ad0

[sq_02] What recent work exists on chain of thought prompting?
  Failed: keyword_coverage=0.75
  Trace: http://localhost:3000/trace/1b3a4a1154b2cd2de373cc346d336cdb

[sq_03] Find papers on mixture of experts models
  Failed: keyword_coverage=0.50
  Trace: http://localhost:3000/trace/40505d38ffc6a7d6482da1fac90e243f

[bi_02] asdkjhasd lkjhaksjdh paper please
  Failed: tool_called=0.00, no_hallucination=0.00
  Trace: http://localhost:3000/trace/7420c8913aa399072d3f6ffcbf6c25c3

